# 02 | Data Preprocessing and Source Table Validation

This notebook prepares clean, consistently typed working copies of the raw traffic, road-network, weather, and segment-status tables. It standardizes timestamps, derives HCMC-local time features, normalizes road attributes, and checks whether missing road attributes can be recovered from the street reference table.

The raw CSV files are read-only inputs. Preprocessed tables are exported as Parquet files under `data/processed` for use by the integration workflow in the next notebook.

## 1. Project Paths and Output Directory

Find the repository root from the current working directory, define the raw and processed data directories, and create the processed directory if it does not already exist.

In [1]:

from pathlib import Path
import pandas as pd
import numpy as np

# Find project root
PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "data" / "raw").exists():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("Project root not found")
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:", PROJECT_ROOT)
print("Raw directory:", RAW_DIR)
print("Processed directory:", PROCESSED_DIR)


Project root: c:\CODE_PROJECTS\KHAI PHA DU LIEU\FinalTermProject\hcmc-traffic-los-forecasting
Raw directory: c:\CODE_PROJECTS\KHAI PHA DU LIEU\FinalTermProject\hcmc-traffic-los-forecasting\data\raw
Processed directory: c:\CODE_PROJECTS\KHAI PHA DU LIEU\FinalTermProject\hcmc-traffic-los-forecasting\data\processed


## 2. Load the Raw Source Datasets

Read the six source CSV files into separate `*_raw` DataFrames. Keeping these raw references separate from the working copies makes the transformations reproducible and avoids modifying the original input data.

In [2]:

# Load original data without modifying raw files

train_raw = pd.read_csv(
    RAW_DIR / "train.csv",
    low_memory=False
)

segments_raw = pd.read_csv(
    RAW_DIR / "segments.csv",
    low_memory=False
)

streets_raw = pd.read_csv(
    RAW_DIR / "streets.csv",
    low_memory=False
)

nodes_raw = pd.read_csv(
    RAW_DIR / "nodes.csv",
    low_memory=False
)

weather_raw = pd.read_csv(
    RAW_DIR / "weather_hcm.csv",
    low_memory=False
)

status_raw = pd.read_csv(
    RAW_DIR / "segment_status.csv",
    low_memory=False
)

print("All raw datasets loaded.")

for name, df in {
    "train": train_raw,
    "segments": segments_raw,
    "streets": streets_raw,
    "nodes": nodes_raw,
    "weather": weather_raw,
    "segment_status": status_raw
}.items():

    print(f"{name}: {df.shape}")


All raw datasets loaded.
train: (33441, 18)
segments: (84633, 11)
streets: (5553, 5)
nodes: (577967, 3)
weather: (7056, 8)
segment_status: (90938, 4)


## 3. Build UTC Observation and Target Timestamps

Create a working copy of the training observations, parse the hour and minute encoded in `period`, and construct a UTC observation timestamp. Define the prediction target timestamp as exactly 30 minutes after the observation time, then validate the parsed timestamps.

In [3]:

# Create a working copy
train = train_raw.copy()

# Extract hour and minute
period_parts = train["period"].str.extract(
    r"^period_(\d{1,2})_(\d{2})$"
)

train["hour_utc"] = pd.to_numeric(
    period_parts[0],
    errors="coerce"
)

train["minute_utc"] = pd.to_numeric(
    period_parts[1],
    errors="coerce"
)

# Build UTC timestamp
train["observation_time_utc"] = pd.to_datetime(
    train["date"].astype(str)
    + " "
    + train["hour_utc"].astype("Int64").astype(str).str.zfill(2)
    + ":"
    + train["minute_utc"].astype("Int64").astype(str).str.zfill(2),
    utc=True,
    errors="coerce"
)

# Validate
assert train["observation_time_utc"].notna().all(), \
    "Invalid observation timestamps detected!"

# Target time
train["target_time_utc"] = (
    train["observation_time_utc"]
    + pd.Timedelta(minutes=30)
)

print("Timestamp conversion successful.")

display(
    train[
        [
            "segment_id",
            "date",
            "period",
            "weekday",
            "observation_time_utc",
            "target_time_utc"
        ]
    ].head(10)
)


Timestamp conversion successful.


,segment_id,date,period,weekday,observation_time_utc,target_time_utc
0,26,2021-04-16,period_0_30,4,2021-04-16 00:30:00+00:00,2021-04-16 01:00:00+00:00
1,33,2020-08-02,period_23_30,6,2020-08-02 23:30:00+00:00,2020-08-03 00:00:00+00:00
2,33,2020-08-03,period_0_00,0,2020-08-03 00:00:00+00:00,2020-08-03 00:30:00+00:00
3,67,2021-03-09,period_9_30,1,2021-03-09 09:30:00+00:00,2021-03-09 10:00:00+00:00
4,67,2021-03-23,period_9_30,1,2021-03-23 09:30:00+00:00,2021-03-23 10:00:00+00:00
5,70,2020-08-02,period_23_30,6,2020-08-02 23:30:00+00:00,2020-08-03 00:00:00+00:00
6,70,2020-08-03,period_0_00,0,2020-08-03 00:00:00+00:00,2020-08-03 00:30:00+00:00
7,71,2020-08-02,period_23_30,6,2020-08-02 23:30:00+00:00,2020-08-03 00:00:00+00:00
8,71,2020-08-03,period_0_00,0,2020-08-03 00:00:00+00:00,2020-08-03 00:30:00+00:00
9,161,2020-09-11,period_0_30,4,2020-09-11 00:30:00+00:00,2020-09-11 01:00:00+00:00


## 4. Derive HCMC-Local Temporal Features

Convert the UTC observation timestamp to the `Asia/Ho_Chi_Minh` timezone. Derive local hour, minute, weekday, month, and weekend indicators, together with cyclical encodings for time of day.

In [4]:

# Convert UTC to HCMC local time
train["observation_time_local"] = (
    train["observation_time_utc"]
    .dt.tz_convert("Asia/Ho_Chi_Minh")
)

# Extract local temporal features
train["hour_local"] = (
    train["observation_time_local"].dt.hour
)

train["minute_local"] = (
    train["observation_time_local"].dt.minute
)

train["weekday_local"] = (
    train["observation_time_local"].dt.day_of_week
)

train["month_local"] = (
    train["observation_time_local"].dt.month
)

train["is_weekend"] = (
    train["weekday_local"] >= 5
).astype("int8")

# Cyclical encoding
hour_decimal = (
    train["hour_local"]
    + train["minute_local"] / 60
)

train["hour_sin"] = np.sin(
    2 * np.pi * hour_decimal / 24
)

train["hour_cos"] = np.cos(
    2 * np.pi * hour_decimal / 24
)

display(
    train[
        [
            "observation_time_utc",
            "observation_time_local",
            "weekday",
            "weekday_local",
            "hour_local",
            "is_weekend",
            "hour_sin",
            "hour_cos"
        ]
    ].head(15)
)


,observation_time_utc,observation_time_local,weekday,weekday_local,hour_local,is_weekend,hour_sin,hour_cos
0,2021-04-16 00:30:00+00:00,2021-04-16 07:30:00+07:00,4,4,7,0,0.923880,-0.382683
1,2020-08-02 23:30:00+00:00,2020-08-03 06:30:00+07:00,6,0,6,0,0.991445,-0.130526
2,2020-08-03 00:00:00+00:00,2020-08-03 07:00:00+07:00,0,0,7,0,0.965926,-0.258819
3,2021-03-09 09:30:00+00:00,2021-03-09 16:30:00+07:00,1,1,16,0,-0.923880,-0.382683
4,2021-03-23 09:30:00+00:00,2021-03-23 16:30:00+07:00,1,1,16,0,-0.923880,-0.382683
5,2020-08-02 23:30:00+00:00,2020-08-03 06:30:00+07:00,6,0,6,0,0.991445,-0.130526
6,2020-08-03 00:00:00+00:00,2020-08-03 07:00:00+07:00,0,0,7,0,0.965926,-0.258819
7,2020-08-02 23:30:00+00:00,2020-08-03 06:30:00+07:00,6,0,6,0,0.991445,-0.130526
8,2020-08-03 00:00:00+00:00,2020-08-03 07:00:00+07:00,0,0,7,0,0.965926,-0.258819
9,2020-09-11 00:30:00+00:00,2020-09-11 07:30:00+07:00,4,4,7,0,0.923880,-0.382683


## 5. Parse and Validate Weather Timestamps

Interpret the weather source timestamps as HCMC local time, localize them to the correct timezone, and convert them to UTC. Check timestamp uniqueness, sampling intervals, missing values, and the ranges of the weather variables.

In [5]:

weather = weather_raw.copy()

# 1. Parse original local timestamps
weather["weather_time_local"] = pd.to_datetime(
    weather["time"],
    errors="coerce"
)

assert weather["weather_time_local"].notna().all(), \
    "Invalid weather timestamps detected!"

# 2. Localize HCMC timezone and convert to UTC
weather["weather_time_local"] = (
    weather["weather_time_local"]
    .dt.tz_localize("Asia/Ho_Chi_Minh")
)

weather["weather_time_utc"] = (
    weather["weather_time_local"]
    .dt.tz_convert("UTC")
)

# 3. Sort chronologically
weather = weather.sort_values(
    "weather_time_utc"
).reset_index(drop=True)

# 4. Check duplicate timestamps
print(
    "Duplicate weather timestamps:",
    weather["weather_time_utc"].duplicated().sum()
)

# 5. Check time intervals
weather["time_gap_minutes"] = (
    weather["weather_time_utc"].diff()
    .dt.total_seconds() / 60
)

print("\nWeather time gap distribution:")
display(
    weather["time_gap_minutes"].describe()
)

# 6. Check missing values
print("\nMissing values:")
display(
    weather.isna().sum().to_frame("Missing")
)

# 7. Check weather feature statistics
weather_features = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "weather_code",
    "wind_speed_10m",
    "cloud_cover"
]

print("\nWeather statistics:")
display(weather[weather_features].describe())

# 8. Display sample
display(
    weather[
        [
            "time",
            "weather_time_local",
            "weather_time_utc"
        ] + weather_features
    ].head(10)
)


Duplicate weather timestamps: 0

Weather time gap distribution:


count    7055.0
mean       60.0
std         0.0
min        60.0
25%        60.0
50%        60.0
75%        60.0
max        60.0
Name: time_gap_minutes, dtype: float64


Missing values:


,Missing
time,0
temperature_2m,0
relative_humidity_2m,0
precipitation,0
rain,0
weather_code,0
wind_speed_10m,0
cloud_cover,0
weather_time_local,0
weather_time_utc,0



Weather statistics:


,temperature_2m,relative_humidity_2m,precipitation,rain,weather_code,wind_speed_10m,cloud_cover
count,7056.000000,7056.000000,7056.000000,7056.000000,7056.000000,7056.000000,7056.00000
mean,27.257129,77.199830,0.248724,0.248724,18.001559,8.312798,73.28784
std,2.874512,16.699276,0.850536,0.850536,23.947713,4.354238,31.53375
min,19.400000,24.000000,0.000000,0.000000,0.000000,0.000000,0.00000
25%,25.100000,67.000000,0.000000,0.000000,2.000000,5.200000,52.00000
50%,26.700000,81.000000,0.000000,0.000000,3.000000,7.500000,89.00000
75%,29.200000,91.000000,0.100000,0.100000,51.000000,10.500000,98.00000
max,36.900000,100.000000,22.300000,22.300000,65.000000,28.900000,100.00000


,time,weather_time_local,weather_time_utc,temperature_2m,relative_humidity_2m,precipitation,rain,weather_code,wind_speed_10m,cloud_cover
0,2020-07-03T00:00,2020-07-03 00:00:00+07:00,2020-07-02 17:00:00+00:00,25.6,96,0.8,0.8,53,6.1,99
1,2020-07-03T01:00,2020-07-03 01:00:00+07:00,2020-07-02 18:00:00+00:00,25.6,96,1.2,1.2,55,4.8,98
2,2020-07-03T02:00,2020-07-03 02:00:00+07:00,2020-07-02 19:00:00+00:00,25.3,96,1.1,1.1,55,5.7,97
3,2020-07-03T03:00,2020-07-03 03:00:00+07:00,2020-07-02 20:00:00+00:00,25.1,97,1.3,1.3,61,5.9,100
4,2020-07-03T04:00,2020-07-03 04:00:00+07:00,2020-07-02 21:00:00+00:00,25.1,97,0.5,0.5,53,6.4,100
5,2020-07-03T05:00,2020-07-03 05:00:00+07:00,2020-07-02 22:00:00+00:00,25.1,97,1.1,1.1,55,6.0,100
6,2020-07-03T06:00,2020-07-03 06:00:00+07:00,2020-07-02 23:00:00+00:00,25.2,96,1.0,1.0,55,6.3,100
7,2020-07-03T07:00,2020-07-03 07:00:00+07:00,2020-07-03 00:00:00+00:00,25.9,93,0.8,0.8,53,5.6,100
8,2020-07-03T08:00,2020-07-03 08:00:00+07:00,2020-07-03 01:00:00+00:00,26.6,91,0.1,0.1,51,11.4,99
9,2020-07-03T09:00,2020-07-03 09:00:00+07:00,2020-07-03 02:00:00+00:00,27.4,84,0.8,0.8,53,14.3,99


## 6. Parse and Validate Segment-Status Timestamps

Parse `updated_at` as UTC, ensure velocity is numeric, sort records by segment and timestamp, and check duplicate segment-time keys and basic velocity quality statistics.

In [6]:

status = status_raw.copy()

# 1. Parse timestamp as UTC
status["updated_at_utc"] = pd.to_datetime(
    status["updated_at"],
    utc=True,
    errors="coerce"
)

assert status["updated_at_utc"].notna().all(), \
    "Invalid status timestamps detected!"

# 2. Ensure numeric velocity
status["velocity"] = pd.to_numeric(
    status["velocity"],
    errors="coerce"
)

# 3. Sort by segment and timestamp
status = status.sort_values(
    ["segment_id", "updated_at_utc"]
).reset_index(drop=True)

# 4. Check duplicate keys
print(
    "Duplicate segment-time keys:",
    status.duplicated(
        ["segment_id", "updated_at_utc"]
    ).sum()
)

# 5. Check velocity distribution
print("\nVelocity statistics:")
display(status["velocity"].describe())

print("\nVelocity quality checks:")

print(
    "Missing velocity:",
    status["velocity"].isna().sum()
)

print(
    "Zero velocity:",
    (status["velocity"] == 0).sum()
)

print(
    "Negative velocity:",
    (status["velocity"] < 0).sum()
)

print(
    "Velocity > 100:",
    (status["velocity"] > 100).sum()
)

# 6. Display sample
display(
    status[
        [
            "segment_id",
            "updated_at",
            "updated_at_utc",
            "velocity"
        ]
    ].head(10)
)


Duplicate segment-time keys: 0

Velocity statistics:


count    90938.000000
mean        20.754844
std         19.567112
min          0.000000
25%          2.000000
50%         17.000000
75%         34.000000
max        200.000000
Name: velocity, dtype: float64


Velocity quality checks:
Missing velocity: 0
Zero velocity: 448
Negative velocity: 0
Velocity > 100: 151


,segment_id,updated_at,updated_at_utc,velocity
0,26,2021-04-16T00:55:31.333Z,2021-04-16 00:55:31.333+00:00,117
1,33,2020-08-02T23:56:28.019Z,2020-08-02 23:56:28.019+00:00,34
2,33,2020-08-02T23:59:28.137Z,2020-08-02 23:59:28.137+00:00,26
3,33,2020-08-03T00:02:28.070Z,2020-08-03 00:02:28.070+00:00,2
4,33,2020-08-03T00:05:28.204Z,2020-08-03 00:05:28.204+00:00,35
5,33,2020-08-03T00:08:28.470Z,2020-08-03 00:08:28.470+00:00,19
6,33,2020-08-03T00:11:28.067Z,2020-08-03 00:11:28.067+00:00,2
7,33,2020-08-03T00:14:28.052Z,2020-08-03 00:14:28.052+00:00,23
8,33,2020-08-03T00:17:28.048Z,2020-08-03 00:17:28.048+00:00,20
9,33,2020-08-03T00:20:27.545Z,2020-08-03 00:20:27.545+00:00,32


## 7. Clean Road-Network Reference Tables

Create working copies of the segment, street, and node tables. Validate their primary keys, normalize text fields, convert designated attributes to numeric types, and inspect missing values, coordinates, road lengths, and speed-limit fields.

In [7]:

# 1. Create working copies
segments = segments_raw.copy()
streets = streets_raw.copy()
nodes = nodes_raw.copy()

# 2. Validate primary keys
assert segments["_id"].is_unique, "Duplicate segment IDs!"
assert streets["_id"].is_unique, "Duplicate street IDs!"
assert nodes["_id"].is_unique, "Duplicate node IDs!"

# 3. Register all tables
tables = {
    "segments": segments,
    "streets": streets,
    "nodes": nodes
}

# 4. Normalize text columns
text_columns = {
    "segments": ["street_name", "street_type"],
    "streets": ["name", "type"]
}

for table_name, columns in text_columns.items():

    for col in columns:

        tables[table_name][col] = (
            tables[table_name][col]
            .astype("string")
            .str.strip()
            .replace("", pd.NA)
        )

# 5. Convert numeric columns
numeric_columns = {
    "segments": [
        "length",
        "max_velocity",
        "street_level",
        "street_id",
        "s_node_id",
        "e_node_id"
    ],
    "streets": [
        "level",
        "max_velocity"
    ],
    "nodes": [
        "long",
        "lat"
    ]
}

for table_name, columns in numeric_columns.items():

    for col in columns:

        tables[table_name][col] = pd.to_numeric(
            tables[table_name][col],
            errors="coerce"
        )

# 6. Update all DataFrame references
segments = tables["segments"]
streets = tables["streets"]
nodes = tables["nodes"]

# 7. Check missing values
for name, df in tables.items():

    print(f"\n{name.upper()} - Missing values")

    display(
        df.isna().sum().to_frame("Missing")
    )

# 8. Check coordinate ranges
print("\nNode coordinate statistics:")

display(
    nodes[["long", "lat"]].describe()
)

# 9. Check road length
print("\nRoad length statistics:")

display(
    segments["length"].describe()
)

# 10. Check maximum velocity
print("\nSegments max_velocity statistics:")

display(
    segments["max_velocity"].describe()
)

print("\nStreets max_velocity statistics:")

display(
    streets["max_velocity"].describe()
)

print("\nROAD DATA PREPROCESSING COMPLETED!")



SEGMENTS - Missing values


,Missing
_id,0
created_at,0
updated_at,0
s_node_id,0
e_node_id,0
length,0
street_id,0
max_velocity,74762
street_level,0
street_name,152



STREETS - Missing values


,Missing
_id,0
level,0
max_velocity,4566
name,1605
type,0



NODES - Missing values


,Missing
_id,0
long,0
lat,0


... (5 outputs left out)

count    9871.000000
mean       54.503596
std        16.553367
min        10.000000
25%        40.000000
50%        60.000000
75%        60.000000
max       120.000000
Name: max_velocity, dtype: float64


Streets max_velocity statistics:


count    987.000000
mean      58.232016
std       16.046644
min       10.000000
25%       50.000000
50%       60.000000
75%       60.000000
max      120.000000
Name: max_velocity, dtype: float64


ROAD DATA PREPROCESSING COMPLETED!


## 8. Audit Road-Attribute Recovery

Join segments to their parent street attributes to determine whether missing segment-level maximum velocity or street names can be recovered. Also audit zero and negative road lengths. This step is diagnostic: it checks recovery opportunities rather than filling values without supporting evidence.

In [8]:

# 1. Prepare street lookup
street_lookup = streets[
    ["_id", "name", "max_velocity"]
].rename(columns={
    "_id": "street_id",
    "name": "street_name_from_streets",
    "max_velocity": "street_max_velocity"
})

# 2. Join segments with streets
segments_audit = segments.merge(
    street_lookup,
    on="street_id",
    how="left",
    validate="many_to_one"
)

# 3. Audit max_velocity recovery
missing_segment_velocity = (
    segments_audit["max_velocity"].isna()
)

street_velocity_available = (
    segments_audit["street_max_velocity"].notna()
)

both_velocity_available = (
    ~missing_segment_velocity
    & street_velocity_available
)

velocity_recovery = {
    "Total segments": len(segments_audit),

    "Missing segment max_velocity":
        int(missing_segment_velocity.sum()),

    "Missing segment velocity but street velocity available":
        int(
            (
                missing_segment_velocity
                & street_velocity_available
            ).sum()
        ),

    "Both velocity values available":
        int(both_velocity_available.sum()),

    "Matching velocity values":
        int(
            (
                segments_audit.loc[
                    both_velocity_available,
                    "max_velocity"
                ]
                ==
                segments_audit.loc[
                    both_velocity_available,
                    "street_max_velocity"
                ]
            ).sum()
        )
}

print("[MAX VELOCITY RECOVERY AUDIT]")

for key, value in velocity_recovery.items():
    print(f"{key}: {value}")

# 4. Audit street name recovery
missing_segment_name = (
    segments_audit["street_name"].isna()
)

street_name_available = (
    segments_audit["street_name_from_streets"].notna()
)

print("\n[STREET NAME RECOVERY AUDIT]")

print(
    "Missing segment street_name:",
    missing_segment_name.sum()
)

print(
    "Missing segment name but street name available:",
    (
        missing_segment_name
        & street_name_available
    ).sum()
)

# 5. Audit road length
print("\n[ROAD LENGTH AUDIT]")

print(
    "Zero length:",
    (segments["length"] == 0).sum()
)

print(
    "Negative length:",
    (segments["length"] < 0).sum()
)

print(
    "Maximum length:",
    segments["length"].max()
)

print("\nROAD ATTRIBUTE AUDIT COMPLETED!")


[MAX VELOCITY RECOVERY AUDIT]
Total segments: 84633
Missing segment max_velocity: 74762
Missing segment velocity but street velocity available: 0
Both velocity values available: 9871
Matching velocity values: 9871

[STREET NAME RECOVERY AUDIT]
Missing segment street_name: 152
Missing segment name but street name available: 0

[ROAD LENGTH AUDIT]
Zero length: 154
Negative length: 0
Maximum length: 1622

ROAD ATTRIBUTE AUDIT COMPLETED!


## 9. Export Preprocessed Source Tables

Save the prepared working tables as Parquet files in `data/processed`. These intermediate outputs preserve the cleaned source-level tables for downstream integration and validation.

In [9]:

preprocessed_tables = {
    "train_preprocessed": train,
    "segments_preprocessed": segments,
    "streets_preprocessed": streets,
    "nodes_preprocessed": nodes,
    "weather_preprocessed": weather,
    "segment_status_preprocessed": status
}

print("Saving preprocessed datasets...\n")

for name, df in preprocessed_tables.items():

    output_path = PROCESSED_DIR / f"{name}.parquet"

    df.to_parquet(
        output_path,
        index=False,
        engine="pyarrow"
    )

    print(
        f"{name}: {df.shape} -> {output_path.name}"
    )

print("\nALL PREPROCESSED DATASETS SAVED SUCCESSFULLY!")


Saving preprocessed datasets...

train_preprocessed: (33441, 30) -> train_preprocessed.parquet
segments_preprocessed: (84633, 11) -> segments_preprocessed.parquet
streets_preprocessed: (5553, 5) -> streets_preprocessed.parquet
nodes_preprocessed: (577967, 3) -> nodes_preprocessed.parquet
weather_preprocessed: (7056, 11) -> weather_preprocessed.parquet
segment_status_preprocessed: (90938, 5) -> segment_status_preprocessed.parquet

ALL PREPROCESSED DATASETS SAVED SUCCESSFULLY!
